# Lab 12: LSTM Model for Time-Series Forecasting

**Student Name:** Yousaf Tahir  
**Registration Number:** 22JZELE0479  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Nowshera Campus  



## Learning Objectives
- Set the working directory and import Scikit-learn, TensorFlow/Keras, and custom time-series utilities.
- Define an LSTM neural network architecture for sequential forecasting data.
- Configure optimizer, model checkpoints, and training monitor callbacks.
- Load training, validation, and testing CSV files for time-series prediction.
- Evaluate LSTM forecasting output using MAE, MSE, RMSE, R2 score, and explained variance.


## Section 1: Working Directory and Library Imports
This section sets the Lab 12 working directory and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utility functions.


In [1]:
import os
os.chdir('D:\Machine Learning\ML Lab\Lab 12')

In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [3]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [4]:
def create_lstm():
    input_data = Input(shape=(time_steps, num_features))
    lstm_layer1 = LSTM(8, return_sequences=True)(input_data)
    lstm_layer2 = LSTM(20)(lstm_layer1)
    x = Flatten()(lstm_layer2)
    output_data = Dense(1)(x)
    model = Model(input_data, output_data)
    return model

## Section 2: Model Parameters and LSTM Architecture
The following cells define time steps, feature count, and the LSTM model architecture used for sequential forecasting.


In [5]:
model1 = create_lstm()
model1.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 21)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 24, 8)          │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 20)             │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,301 (12.89 KB)

 Trainable params: 3,301 (12.89 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [7]:
checkpoints = r'D:\Machine Learning\ML Lab\Lab 12\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'D:\Machine Learning\ML Lab\Lab 12'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [8]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

## Section 3: Checkpoints, Callbacks, and Training Configuration
This section prepares checkpoint paths, output directories, training monitor callbacks, optimizer settings, and model compilation.


In [9]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =create_lstm()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [10]:
path_dataset = r'D:\Machine Learning\ML Lab\Lab 12'

path_tr = os.path.join(path_dataset, 'AEP_train.csv')
path_v = os.path.join(path_dataset, 'AEP_validation.csv')
path_te = os.path.join(path_dataset, 'AEP_test.csv')
path_hourly = os.path.join(path_dataset, 'AEP_hourly.csv')

for path in [path_tr, path_v, path_te, path_hourly]:
    print(path, "=>", os.path.exists(path))

df_tr = pd.read_csv(path_tr)
train_set = df_tr.values

df_v = pd.read_csv(path_v)
validation_set = df_v.values

df_te = pd.read_csv(path_te)
test_set = df_te.values

df_hourly = pd.read_csv(path_hourly)

train_set.shape, validation_set.shape, test_set.shape, df_hourly.shape

D:\Machine Learning\ML Lab\Lab 12\AEP_train.csv => True
D:\Machine Learning\ML Lab\Lab 12\AEP_validation.csv => True
D:\Machine Learning\ML Lab\Lab 12\AEP_test.csv => True
D:\Machine Learning\ML Lab\Lab 12\AEP_hourly.csv => True


((84907, 21), (24259, 21), (12130, 21), (121273, 2))

In [11]:
time_steps=24
num_features=21

In [12]:
start = time.time()

train_X, train_y = univariate_multi_step(
    train_set,
    time_steps,
    target_col=0,
    target_len=1
)

validation_X, validation_y = univariate_multi_step(
    validation_set,
    time_steps,
    target_col=0,
    target_len=1
)

test_X, test_y = univariate_multi_step(
    test_set,
    time_steps,
    target_col=0,
    target_len=1
)

print('Time Consumed', time.time() - start, "sec")

Time Consumed 0.3776381015777588 sec


## Section 4: Dataset Loading and Forecast Preparation
The following cells load train, validation, and test CSV files, then prepare the data for LSTM forecasting input/output format.


In [13]:
epochs = 10
verbose = 1
batch_size = 32

History = model.fit(
    train_X,
    train_y,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(validation_X, validation_y),
    callbacks=callbacks,
    verbose=verbose
)

Epoch 1/10
2646/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0384 - mae: 0.0384 - mape: 677.2867
Epoch 1: val_loss improved from None to 0.01867, saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0001-loss0.02.h5



Epoch 1: finished saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0001-loss0.02.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - loss: 0.0383 - mae: 0.0383 - mape: 675.6271 - val_loss: 0.0187 - val_mae: 0.0187 - val_mape: 9.4192
Epoch 2/10
2644/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0164 - mae: 0.0164 - mape: 518.4185
Epoch 2: val_loss improved from 0.01867 to 0.01505, saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0002-loss0.02.h5



Epoch 2: finished saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0002-loss0.02.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - loss: 0.0164 - mae: 0.0164 - mape: 516.7564 - val_loss: 0.0151 - val_mae: 0.0151 - val_mape: 7.7892
Epoch 3/10
2644/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0120 - mae: 0.0120 - mape: 179.3001
Epoch 3: val_loss improved from 0.01505 to 0.01131, saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0003-loss0.01.h5



Epoch 3: finished saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0003-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - loss: 0.0120 - mae: 0.0120 - mape: 178.7320 - val_loss: 0.0113 - val_mae: 0.0113 - val_mape: 5.6847
Epoch 4/10
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0112 - mae: 0.0112 - mape: 260.4667
Epoch 4: val_loss improved from 0.01131 to 0.01007, saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0004-loss0.01.h5



Epoch 4: finished saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0004-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - loss: 0.0112 - mae: 0.0112 - mape: 260.4667 - val_loss: 0.0101 - val_mae: 0.0101 - val_mape: 4.3204
Epoch 5/10
2651/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0107 - mae: 0.0107 - mape: 187.5592
Epoch 5: val_loss improved from 0.01007 to 0.00973, saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0005-loss0.01.h5



Epoch 5: finished saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0005-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 16s 6ms/step - loss: 0.0107 - mae: 0.0107 - mape: 187.4504 - val_loss: 0.0097 - val_mae: 0.0097 - val_mape: 4.0989
Epoch 6/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0102 - mae: 0.0102 - mape: 167.0220
Epoch 6: val_loss improved from 0.00973 to 0.00931, saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0006-loss0.01.h5



Epoch 6: finished saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0006-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - loss: 0.0102 - mae: 0.0102 - mape: 166.9874 - val_loss: 0.0093 - val_mae: 0.0093 - val_mape: 4.3436
Epoch 7/10
2645/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0099 - mae: 0.0099 - mape: 106.7619
Epoch 7: val_loss did not improve from 0.00931
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - loss: 0.0099 - mae: 0.0099 - mape: 106.4658 - val_loss: 0.0102 - val_mae: 0.0102 - val_mape: 4.4667
Epoch 8/10
2645/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0093 - mae: 0.0093 - mape: 330.4374
Epoch 8: val_loss did not improve from 0.00931
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - loss: 0.0093 - mae: 0.0093 - mape: 329.5027 - val_loss: 0.0096 - val_mae: 0.0096 - val_mape: 4.2849
Epoch 9/10
2646/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0089 - mae: 0.0089 - mape: 399.4630
Epoch 9: val_loss improved from 0.00931 to 0.00874, saving model to D:\Mach


Epoch 9: finished saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0009-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - loss: 0.0089 - mae: 0.0089 - mape: 398.4812 - val_loss: 0.0087 - val_mae: 0.0087 - val_mape: 3.9902
Epoch 10/10
2646/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0085 - mae: 0.0085 - mape: 187.4521
Epoch 10: val_loss improved from 0.00874 to 0.00772, saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0010-loss0.01.h5



Epoch 10: finished saving model to D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0010-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - loss: 0.0085 - mae: 0.0085 - mape: 186.9946 - val_loss: 0.0077 - val_mae: 0.0077 - val_mape: 3.2515


In [14]:
import os
import pickle

path_dataset = r'D:\Machine Learning\ML Lab\Lab 12'
path_scaler = os.path.join(path_dataset, 'AEP_Scaler.pkl')

print(os.path.exists(path_scaler))

with open(path_scaler, 'rb') as f:
    scaler = pickle.load(f)

True


c:\Users\Yousaf Tahir\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [15]:
model_path = r'D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0010-loss0.01.h5'

model = load_model(model_path, compile=False)

y_pred_scaled = model.predict(test_X)

print("y_pred_scaled.shape =", y_pred_scaled.shape)
print("test_y.shape =", test_y.shape)

def inverse_target(scaled_values, scaler, target_col=0, num_features=21):
    scaled_values = scaled_values.reshape(-1)

    dummy = np.zeros((scaled_values.shape[0], num_features))
    dummy[:, target_col] = scaled_values

    unscaled = scaler.inverse_transform(dummy)

    return unscaled[:, target_col].reshape(-1, 1)

y_pred = inverse_target(
    y_pred_scaled,
    scaler,
    target_col=0,
    num_features=21
)

y_test_unscaled = inverse_target(
    test_y,
    scaler,
    target_col=0,
    num_features=21
)

# Mean Absolute Error (MAE)
MAE = np.mean(np.abs(y_pred - y_test_unscaled))
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(np.abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.mean(np.square(y_pred - y_test_unscaled))
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squared Error (RMSE)
RMSE = np.sqrt(MSE)
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape = ', y_test_unscaled.shape)
print('y_pred.shape = ', y_pred.shape)

379/379 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
y_pred_scaled.shape = (12105, 1)
test_y.shape = (12105, 1)
Mean Absolute Error (MAE): 125.2
Median Absolute Error (MedAE): 100.05
Mean Squared Error (MSE): 26510.53
Root Mean Squared Error (RMSE): 162.82
Mean Absolute Percentage Error (MAPE): 0.86 %
Median Absolute Percentage Error (MDAPE): 0.69 %


y_test_unscaled.shape =  (12105, 1)
y_pred.shape =  (12105, 1)


In [16]:
checkpoints = r'D:\Machine Learning\ML Lab\Lab 12\\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0010-loss0.01.h5'
start_epoch= 34

## Section 5: Prediction and Model Evaluation
The final cells generate predictions and calculate regression metrics to evaluate the forecasting model performance.


In [17]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
model_load = r'D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0010-loss0.01.h5'
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model_load))
    model = load_model(model_load, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )


    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading D:\Machine Learning\ML Lab\Lab 12\\E1-cp-0010-loss0.01.h5...
[INFO] old learning rate: 9.999999747378752e-05
[INFO] new learning rate: 9.999999747378752e-05


In [19]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
2650/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0073 - mae: 0.0073 - mape: 166.7622
Epoch 1: val_loss improved from None to 0.00712, saving model to D:\Machine Learning\ML Lab\Lab 12\\E2-cp-0001-loss0.01.h5



Epoch 1: finished saving model to D:\Machine Learning\ML Lab\Lab 12\\E2-cp-0001-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - loss: 0.0073 - mae: 0.0073 - mape: 166.6031 - val_loss: 0.0071 - val_mae: 0.0071 - val_mape: 3.0111
Epoch 2/10
2651/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0072 - mae: 0.0072 - mape: 210.6734
Epoch 2: val_loss did not improve from 0.00712
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - loss: 0.0072 - mae: 0.0072 - mape: 210.5506 - val_loss: 0.0072 - val_mae: 0.0072 - val_mape: 3.0468
Epoch 3/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0072 - mae: 0.0072 - mape: 186.1138
Epoch 3: val_loss did not improve from 0.00712
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - loss: 0.0072 - mae: 0.0072 - mape: 186.0748 - val_loss: 0.0072 - val_mae: 0.0072 - val_mape: 3.1171
Epoch 4/10
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0071 - mae: 0.0071 - mape: 164.5441
Epoch 4: val_loss did not improve from 0.00712
2653/2653 ━━━━━━━━━━━━━━━━━━


Epoch 10: finished saving model to D:\Machine Learning\ML Lab\Lab 12\\E2-cp-0010-loss0.01.h5
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - loss: 0.0070 - mae: 0.0070 - mape: 89.1392 - val_loss: 0.0070 - val_mae: 0.0070 - val_mape: 3.0428


In [20]:
model = load_model(r'D:\Machine Learning\ML Lab\Lab 12\\E2-cp-0010-loss0.01.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Mean Absolute Error (MAE): 114.28
Median Absolute Error (MedAE): 89.78
Mean Squared Error (MSE): 22997.58
Root Mean Squared Error (RMSE): 151.65
Mean Absolute Percentage Error (MAPE): 0.78 %
Median Absolute Percentage Error (MDAPE): 0.62 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


## Final Conclusion
In this lab, an LSTM neural network was implemented for time-series forecasting. The notebook demonstrates sequence-model architecture, callback-based training, data preparation, and regression-based model evaluation.
